# Exercise 4.2

__Location 1__  
*  requests: $\lambda = 3 \space (Y_1)$
*  returns: $\lambda = 3 \space (X_1)$

__Location 2__  
* requests: $\lambda = 4 \space (Y_2)$
* returns: $\lambda = 2 \space (X_2)$

__Value Function:__  
$
v_\pi(s) = \sum_a \pi(a \mid s) \sum_{s',r} p(s', r \mid s, a)\left[ r + \gamma v(s') \right]
$

Taking the maximal at each iteration -->  
  
$\pi'(s) = \max_aq_\pi(s,a)$  
$\quad = \max_a(\sum_{s',r} p(s', r \mid s, a)\left[ r + \gamma v(s') \right])$  

__Actions (A):__  
The Action $(A = a)$ is the number of cars transfered between locations.  
*  Maximum 5 cars can be transfered 
*  Cost: 2$ per car   
  
__Joint distribtuion of (s', r|s, a)__

Here we calculate the joint probability of returns and rentals given the action a (transfers)
1. Calculate $p(X1)p(X2)p(Y1)p(Y2)$  
2. Calculate $r = min(Y1, n1)*10 + min(Y2, n2)*10 - a*2$  

Whithout giving it too much thought, this is an expensiver calucation.  
* Requests $Y \in [0, 20]$
* Returns in $X in [0, 20]$
* $\implies 21*21*21*21 = 21^4 = 194,481 \text{ combinations of probabilities}$ for each action  
* Precompute 4 dictionaries of PMF calculations for each value 0-20 for each random variable $X_1, X_2, Y_1, Y_2$ 

* Finish ca
  
__The Calculations:__  
Need to calculate $\sum_{s',r} p(s', r \mid s, a)\left[ r + \gamma v(s') \right]$


__The Calculations:__  
In this example we are using the __Bellman optimality update__.  
Here we maintain:
* a table of value function estimates $V(s)$ with the goal of converging on the Value function  
where $s = (n1, n2) \text{  and n1,n2} \in [0, 20]$    
* $\pi(a|s)$ the policy table 

__Recall:__ 
* $pi_{*}(s) = \arg\max_a(q_*(s,a))$ 
* $V_*(s) = \max_a(q_*(s,a))$

__Algorithm:__  

For each State (s):

1. calculate $q(s,a) = \sum_{s',r} p(s', r \mid s, a)\left[ r + \gamma v(s') \right]$  
Each iteration has x1, x2, y1, y2 and is conditioned on $s = (n1, n2)$ and the action $a$  
let:  
   * $\tilde{n_1} = n_1 - a$
   * $\tilde{n_2} = n_2 + a$
   * Where a > 0 if moving cars from 1 to 2 and a < 0 if moving cars from 2 to 1 

    Calculate $v(s')$:
    * $m_1 = $\tilde{n_1} + x1 - y1$
    * $m_2 = $\tilde{n_2} + x2 - y2$  
    * $v(s') = v((m_1, m_2))$


1. Updates:  
   * $pi_{*}(s) = \arg\max_a(q_*(s,a))$ 
   * $V_*(s) = \max_a(q_*(s,a))$


__Convergences:__  
Repeat the core algorithm until after an epoch unill:  
* $\max(|V_k(s) - V_{k+1}(s)|) < \epsilon$ for all $s \in S$



In [130]:
from itertools import product
from statsmodels.distributions.discrete import poisson as poisson


In [131]:
def get_combinations(*vectors):
    return list(product(*vectors))

env_combinations = get_combinations(range(21), range(21), range(21), range(21))
len(env_combinations)

194481

In [132]:
env_combinations[10017:10022]

[(1, 1, 15, 0), (1, 1, 15, 1), (1, 1, 15, 2), (1, 1, 15, 3), (1, 1, 15, 4)]

In [133]:
pmfY1 = poisson.pmf(range(21), 3)
pmfY2 = poisson.pmf(range(21), 4)
pmfX1 = poisson.pmf(range(21), 3) 
pmfX2 = poisson.pmf(range(21), 4)

# V[i][j] => the value for state (i, j)  i & j in (0, 20) => a 21X21 matrix 
V = [[0.0 for _ in range(21)] for _ in range(21)]
pi = [[0 for _ in range(21)] for _ in range(21)]
states = get_combinations(range(21), range(21))

 $q_\pi(s,a) = \sum_{s',r} p(s', r \mid s,a)\left[r + \gamma v_\pi(s')\right]$

Given state (n1, n2):
for each environment event (y1, y2, x1, x2):

1. Apply action
   n1_a = n1 - a
   n2_a = n2 + a

2. If action infeasible, skip or return 0
   e.g. n1_a < 0 or n2_a < 0 or n1_a > 20 or n2_a > 20

3. Event probabilities
   p_req1 = pmfY1[y1]
   p_req2 = pmfY2[y2]
   p_ret1 = pmfX1[x1]
   p_ret2 = pmfX2[x2]

   jointProb = p_req1 * p_req2 * p_ret1 * p_ret2

4. Actual rentals
   rent1 = min(y1, n1_a)
   rent2 = min(y2, n2_a)

5. Reward
   r = 10*rent1 + 10*rent2 - 2*abs(a)

6. Next state
   n1_next = min(20, n1_a - rent1 + x1)
   n2_next = min(20, n2_a - rent2 + x2)

7. Partial Bellman contribution
   partialValue = jointProb * (r + gamma * V[n1_next][n2_next])

In [134]:
def getPartialBellmanStateAction(state, env, action, V_ref = None, gamma = .9, maxCars = 20):
    '''
    Calculate the partial Bellman state action given the current state (s) and 
    the sample of environmental dynamics (env).  

    This is the solution to p(s', r|s, a) * [r + gamma * V(s')] 

    Inputs:
    state: [n1, n2]: the starting state, number of cars in location 1 and 2.  
    env: [y1, y2, x1, x2], the envirnmental dynamics describing requests y1, y2 and returns x1, x2
    action: The number of cars moving from loc1 to loc2 or vice versa.  Range [-5, 5] (could be larger)  
    (-) -> move from 2 to 1, (+) -> move from 1 to 2

    Returns:
    value 
    '''
    #For Dask
    if V_ref is None:
        V_ref = V

    # Collect the variables
    n1 = state[0]
    n2 = state[1]
    y1 = env[0]
    y2 = env[1] 
    x1 = env[2]
    x2 = env[3]

    # Apply action
    n1_a = n1 - action
    n2_a = n2 + action

    # Constrain to post action in 0 and 20
    if n1_a < 0 or n2_a < 0 or n1_a > maxCars or n2_a > maxCars:
        return(0)
    
    # Calculate the joint probability of env probabilies 
    p_req1 = pmfY1[y1]
    p_req2 = pmfY2[y2]
    p_ret1 = pmfX1[x1]
    p_ret2 = pmfX2[x2]
    jointProb = p_req1 * p_req2 * p_ret1 * p_ret2

    # Actual rentals
    rent1 = min(y1, n1_a)
    rent2 = min(y2, n2_a)

    #  Reward
    r = 10*rent1 + 10*rent2 - 2*abs(action)

    # Next state
    n1_next = min(maxCars, n1_a - rent1 + x1)
    n2_next = min(maxCars, n2_a - rent2 + x2)

    # Partial Bellman contribution
    partialValue = jointProb * (r + gamma * V_ref[n1_next][n2_next])

    return(partialValue)
        
    

In [135]:
def getBellmanStateActionValue(state, action, V_ref=None):
    if V_ref is None:
        V_ref = V

    bellmanStateActionValue = 0
    for env in env_combinations:
        bellmanStateActionValue += getPartialBellmanStateAction(state, env, action, V_ref=V_ref)

    return(bellmanStateActionValue)


In [136]:
def get_feasible_actions(state, maxMove=5, maxCars=20):
    n1, n2 = state
    actions = []
    for a in range(-maxMove, maxMove + 1):
        n1_a = n1 - a
        n2_a = n2 + a
        if 0 <= n1_a <= maxCars and 0 <= n2_a <= maxCars:
            actions.append(a)
    return actions

In [ ]:
def policy_interation(epsilon = 1e-5):
    maxChange = 1000
    for state in states:
        print(f"\nEvaluation iteration {it}")
        
        print(state)
        action = pi[state[0]][state[1]]
        
        q_sa = getBellmanStateActionValue(state, action)

        V[state[0]][state[1]] = q_sa
        



In [ ]:
policy_interation()

In [139]:


from dask import delayed, compute
from dask.distributed import Client, LocalCluster
import os

n_workers=6
threads_per_worker=1

cluster = LocalCluster(n_workers=6, threads_per_worker=1, processes=True)
client = Client(cluster)

@delayed
def eval_one_state(state, V_snapshot, pi_snapshot):
    action = pi_snapshot[state[0]][state[1]]
    q_sa = getBellmanStateActionValue(state, action, V_snapshot)
    return state, q_sa

def policy_iteration(epsilon=1e-5, max_iter=1000):
    global V

    for iteration in range(max_iter):
        V_snapshot = [row[:] for row in V]
        pi_snapshot = [row[:] for row in pi]

        tasks = [eval_one_state(state, V_snapshot, pi_snapshot) for state in states]
        results = compute(*tasks)

        max_change = 0.0
        new_V = [row[:] for row in V_snapshot]

        for state, q_sa in results:
            n1, n2 = state
            old_v = V_snapshot[n1][n2]
            new_V[n1][n2] = q_sa
            max_change = max(max_change, abs(q_sa - old_v))

        V = new_V
        print(f"iteration={iteration}, max_change={max_change:.8f}")

        if max_change < epsilon:
            break

c:\Users\GregSchwartz\anaconda3\Lib\site-packages\distributed\node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 53908 instead
  warnings.warn(


In [141]:
def policy_improvement(states):
    global pi
    policy_stable = True

    new_pi = [row[:] for row in pi]

    for state in states:
        n1, n2 = state
        old_action = pi[n1][n2]

        best_action = None
        best_value = -float('inf')

        for action in get_feasible_actions(state):
            q_sa = getBellmanStateActionValue(state, action, V_ref=V)

            if q_sa > best_value:
                best_value = q_sa
                best_action = action

        new_pi[n1][n2] = best_action

        if best_action != old_action:
            policy_stable = False

    pi = new_pi
    return policy_stable

In [ ]:
while True:
    policy_iteration(max_iter=2)
    stable = policy_improvement(states)
    if stable:
        break

iteration=0, max_change=69.99999940
iteration=1, max_change=62.99990506


2026-04-20 22:40:13,135 - distributed.scheduler - WARNING - Worker failed to heartbeat for 10114s; attempting restart: <WorkerState 'tcp://127.0.0.1:53946', name: 5, status: running, memory: 0, processing: 0>
2026-04-20 22:40:21,844 - distributed.nanny - WARNING - Restarting worker


In [125]:
import os
os.cpu_count()

12

In [100]:
def getArgValueMax(state, maxMove=5, maxCars=20):
    best_action = None
    best_value = -float('inf')

    feasible_actions = get_feasible_actions(state, maxMove, maxCars)

    for action in feasible_actions:
        q_sa = getBellmanStateActionValue(state, action)
        if q_sa > best_value:
            best_value = q_sa
            best_action = action
        
        print(action, q_sa)


    print(best_action, best_value, "States:", feasible_actions)
    pi[state[0]][state[1]] = best_action
    V[state[0]][state[1]] = best_value    

In [103]:
getArgValueMax((15,15))

-5 59.95950879215962
-4 61.98984433826072
-3 64.00365004664705
-2 66.01484951984874
-1 68.02676528196002
0 70.03640200572583
1 68.03880325295331
2 66.03238878889465
3 64.02090671575188
4 62.00971252740956
5 60.00027959094482
0 70.03640200572583 States: [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5]


In [96]:
state

(1, 0)

In [94]:
pi[10]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 69.95517612771438, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [95]:
V[10]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 69.95517612771438, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [74]:
env = (2,3,2,3)
state = (5, 5)

getPartialBellmanStateAction(state, env, 2)


0.0881287397654725

In [ ]:
getBellmanStateActionValue(state,)

In [76]:
action = 0
state = (1,0)
bellmanStateActionValue = 0
for env in env_combinations:
    bellmanStateActionValue += getPartialBellmanStateAction(state, env, action)

bellmanStateActionValue

9.502129279542382

In [77]:
getBellmanStateActionValue(state, action)

9.502129279542382

In [ ]:
s = (0, 0) 

a = 0 

# Y: requests, X: returns  

n1 = s[0]
n2 = s[1]

n1_a = n1 - a 
n2_a = n2 + a

n1_return = 2
n2_return = 0  


TypeError: 'int' object is not iterable

In [31]:
cumSum = 0
for jointX in jointDist:
    y1 = jointX[0]
    y2 = jointX[1]
    x1 = jointX[2]
    x2 = jointX[3]
    cumSum += pmfY1[y1] * pmfY2[y2] * pmfX1[x1] * pmfX2[x2]


In [32]:
cumSum

0.999999996130165

In [21]:
from math import exp, factorial
lmda = 3
n = 0
p = (lmda**n / factorial(n)) * exp(-lmda)
p == pmfY1[0]


True

In [16]:
poisson.pmf(2, 2)

0.2706705664732254

In [2]:
get_combinations([1, 2, 3], [2, 3, 4], [5, 7, 8])

[(1, 2, 5),
 (1, 2, 7),
 (1, 2, 8),
 (1, 3, 5),
 (1, 3, 7),
 (1, 3, 8),
 (1, 4, 5),
 (1, 4, 7),
 (1, 4, 8),
 (2, 2, 5),
 (2, 2, 7),
 (2, 2, 8),
 (2, 3, 5),
 (2, 3, 7),
 (2, 3, 8),
 (2, 4, 5),
 (2, 4, 7),
 (2, 4, 8),
 (3, 2, 5),
 (3, 2, 7),
 (3, 2, 8),
 (3, 3, 5),
 (3, 3, 7),
 (3, 3, 8),
 (3, 4, 5),
 (3, 4, 7),
 (3, 4, 8)]